# FireCal Harmonized Active Fire Record — quickstart

This notebook loads the harmonized MODIS→VIIRS fire record, plots a long-term series **with its uncertainty**, maps one year, and compares countries and emissions.

* `harmonized_fire_days`: VIIRS-equivalent active-fire days (unique ~1 km cell-days with a nominal/high-confidence detection).
* 2012 onward is **observed** by VIIRS S-NPP; 2003–2011 is **MODIS harmonized** to VIIRS units, with 90% intervals (`*_lo90`, `*_hi90`).
* Methods and validation: `ATBD.md`. Licence: CC BY 4.0. Underlying detections: NASA FIRMS.

```bash
pip install xarray netCDF4 pandas matplotlib
```

In [ ]:
import glob
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

nc = sorted(glob.glob("netcdf/firecal_fire_days_1deg_monthly_*.nc"))[0]
ds = xr.open_dataset(nc)
ds

## 1. Global annual series with uncertainty (from the country table)

In [ ]:
ann = pd.read_csv("tables/country_annual.csv")
world = ann[ann.country == "World"].set_index("year")
fig, ax = plt.subplots(figsize=(9, 4))
modis = world[world.source == "modis_harmonized"]
ax.fill_between(modis.index, modis.harmonized_lo90, modis.harmonized_hi90, alpha=0.3, label="90% interval (MODIS harmonized)")
ax.plot(world.index, world.harmonized_fire_days, "o-", label="harmonized fire-days")
ax.plot(world.index, world.modis_fire_days, "--", alpha=0.6, label="raw MODIS fire-days")
ax.axvline(2011.5, color="grey", lw=0.8)
ax.set_ylabel("fire-days per year"); ax.legend(); ax.set_title("World")
plt.show()

## 2. Map one year

In [ ]:
year = 2015
annual = ds.harmonized_fire_days.sel(time=str(year)).sum("time")
annual.where(annual > 0).plot(figsize=(11, 5), robust=True, cmap="inferno")
plt.title(f"Harmonized fire-days per 1° cell, {year}")
plt.show()

## 3. Seasonal cycle of a region (here: mainland Southeast Asia)

In [ ]:
box = ds.harmonized_fire_days.sel(lat=slice(10, 25), lon=slice(95, 110)).sum(["lat", "lon"])
clim = box.groupby("time.month").mean()
clim.plot(marker="o"); plt.ylabel("mean fire-days per month"); plt.show()

## 4. Countries: trend and emissions

In [ ]:
top = ann[(ann.country != "World") & (ann.year >= 2015)].groupby("country").harmonized_fire_days.mean().nlargest(10)
print(top)
em = ann[ann.country.isin(top.index[:5])].pivot(index="year", columns="country", values="co2_tg")
em.plot(figsize=(9, 4), marker=".")
plt.ylabel("CO₂ from active fires (Tg/yr, conservative)"); plt.show()

## 5. Validation of the harmonization (blind, held-out years)

In [ ]:
val = pd.read_csv("model/validation.csv")
val.pivot_table(index=["train_years", "test_years"], columns="method", values="monthly_median_abs_error_pct")

## 6. How often MODIS detects the fires VIIRS sees

In [ ]:
det = pd.read_csv("model/modis_detection_probability.csv")
for name, g in det.groupby("curve"):
    plt.semilogx(g.viirs_frp_mw, g.observed_probability, "o", label=name)
    plt.semilogx(g.viirs_frp_mw, g.fitted_probability, "-", color=plt.gca().lines[-1].get_color())
plt.xlabel("VIIRS fire radiative power (MW)"); plt.ylabel("P(MODIS detects)"); plt.legend(); plt.show()